# 🏎️ F1 BI — Dashboard Streamlit
## Proyecto Final · Módulo 4: Inteligencia de Negocios y SQL Avanzado

Este notebook levanta el dashboard interactivo en Google Colab usando **Streamlit** expuesto con un túnel **ngrok**.

### Visualizaciones incluidas
| # | Visualización | Técnica SQL |
|:---:|---|---|
| 1 | Campeonato — evolución de puntos por ronda | Window Functions |
| 2 | Estrategia — pit stops y posición de salida | Agregaciones |
| 3 | Dominancia — eficiencia de constructores por era | CTEs anidados |
| 4 | Circuitos — posición mediana de ganadores | PERCENTILE_CONT |
| 5 | Edad — evolución histórica de ganadores | DATE_PART + LAG() |

### Pasos
1. Instalar dependencias
2. Configurar ngrok
3. Subir `dashboard.py` a Colab
4. Lanzar el servidor
5. Abrir la URL pública generada

## 1. Instalación de dependencias

In [ ]:
!pip install streamlit plotly sqlalchemy psycopg2-binary pyngrok -q
print('✅ Dependencias instaladas')

## 2. Variables de entorno

Las credenciales de Aurora se pasan como variables de entorno para que `dashboard.py` las lea con `os.getenv()`. Nunca se hardcodean en el archivo.

In [ ]:
import os
from google.colab import userdata

# Leer credenciales desde los Secrets de Colab
os.environ['F1_HOST'] = userdata.get('F1_HOST')
os.environ['F1_DATA'] = userdata.get('F1_data')
os.environ['F1_USER'] = userdata.get('AURORA_USER')

print('✅ Variables de entorno configuradas')
print(f'   Host: {os.environ["F1_HOST"][:20]}...')

## 3. Subir dashboard.py a Colab

Ejecuta la celda y selecciona el archivo `dashboard.py` desde tu computadora.
Alternativamente, si tu repo está en GitHub puedes clonarlo directamente.

In [ ]:
from google.colab import files

# Opción A: subir el archivo manualmente
uploaded = files.upload()  # Selecciona dashboard.py

# Opción B: clonar desde GitHub (descomenta si usas esta opción)
# !git clone https://github.com/TU_USUARIO/proyecto-final.git
# !cp proyecto-final/dashboard/dashboard.py .

import os
print('✅ Archivo disponible:', os.path.exists('dashboard.py'))

## 4. Configurar ngrok

**ngrok** crea un túnel HTTPS público que expone el servidor Streamlit (que corre en localhost:8501 dentro de Colab) al navegador.

**Obtener token gratuito:** https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok

# Pega tu token de ngrok aquí o guárdalo en los Secrets de Colab
NGROK_TOKEN = userdata.get('NGROK_TOKEN')  # o pega el token directamente
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok configurado')

## 5. Lanzar el dashboard

Esta celda:
1. Levanta Streamlit en el puerto 8501 en background (`&`)
2. Abre un túnel ngrok hacia ese puerto
3. Imprime la URL pública para abrir en el navegador

> **Nota:** El proceso tarda ~10 segundos en iniciar. Si la URL da error al principio, espera unos segundos y recarga.

In [ ]:
import subprocess, time
from pyngrok import ngrok

# 1. Matar procesos previos si los hay
!pkill -f streamlit 2>/dev/null || True
ngrok.kill()
time.sleep(2)

# 2. Lanzar Streamlit en background
proc = subprocess.Popen(
    ['streamlit', 'run', 'dashboard.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(8)  # Esperar a que Streamlit inicie

# 3. Abrir túnel ngrok
tunnel = ngrok.connect(8501, proto='http')
url   = tunnel.public_url

print('=' * 55)
print('  🏎️  F1 Analytics Dashboard')
print('=' * 55)
print(f'  URL pública: {url}')
print('  Abre esa URL en tu navegador')
print('  Para cerrar: ejecuta la celda de abajo')
print('=' * 55)

## 6. Detener el dashboard

Ejecuta esta celda cuando termines para liberar el túnel y el proceso.

In [ ]:
from pyngrok import ngrok

ngrok.kill()
!pkill -f streamlit 2>/dev/null || True
print('✅ Dashboard detenido')

## 7. Solución de problemas comunes

| Problema | Solución |
|---|---|
| La URL da `502 Bad Gateway` | Espera 10 segundos más y recarga |
| `AuthError` en ngrok | Verifica que el token sea correcto |
| `Connection refused` en Aurora | Verifica que las variables de entorno estén seteadas |
| El dashboard muestra gráficas vacías | Verifica que las dimensiones tengan datos con el VALIDATE |
| Colab se desconecta | Vuelve a ejecutar desde la Celda 2 en adelante |